# Vasopressores, diuréticos, diurese e balanço hídrico nas primeiras 24 horas

**Pergunta:** com `inputevents` e `outputevents`, a AUC de "estadia longa" sobe? E quanto dela vem de pacientes que nem chegam a 24 horas na UTI?

**O que entra de novo** (tudo em `[entrada, entrada + 24 h]`)

- `inputevents`: vasopressores (noradrenalina, vasopressina, adrenalina, fenilefrina, dopamina), inotrópicos (dobutamina, milrinona), furosemida (bolus e infusão contínua), volume infundido, ingestão oral e peso.
- `outputevents`: diurese, diurese por quilo e hora, oligúria (< 0,5 mL/kg/h), total de saídas e balanço hídrico.

**Uma correção de método**

O instante de previsão é a marca de 24 horas. Quem sai da UTI antes disso não pode ser previsto nesse instante, e ainda tem menos exames e registros, o que denuncia estadia curta. Por isso, além dos resultados "como antes", este caderno mede a coorte **só com quem ainda está na UTI às 24 horas** (`los_dias ≥ 1`). Essa é a versão que o TCC deve usar.

**Variantes**

| Rótulo | Coorte | Variáveis |
|---|---|---|
| Ref | todas as estadias (como antes) | de antes + 15 novas (chart e labs) |
| E | ainda na UTI às 24 h | as 24 iniciais |
| F | ainda na UTI às 24 h | E + 15 novas (chart e labs) |
| G | ainda na UTI às 24 h | F + inputevents e outputevents |
| E-Y, F-Y, G-Y | o mesmo, só primeira UTI e sem óbitos hospitalares (Yang) | |

**Regras:** o CID só define a coorte; nada depois de `intime + 24 h` entra em X; divisão por paciente (`random_state=42`, 80/20); imputação e one-hot só no treino; hiperparâmetros fixos, iguais aos da etapa anterior.

**Códigos dos itens:** o arquivo `d_items` não está na pasta. Os `itemid` foram conferidos pelos dados (unidade, taxa mediana e categoria). Se você baixar `d_items.csv.gz` para `data/mimic-iv/`, a seção 3 compara os nomes.

**Tempo:** a primeira execução lê `inputevents` e `outputevents` (poucos minutos). O resultado fica em `artefatos_24h_ampliado/tabela_inout_24h.csv`.

Pré-requisito: `artefatos_24h_ampliado/tabela_24h_ampliada.csv`, gerado pelo caderno `variaveis_24h_e_coorte_yang.ipynb`.

**Validação adicional:** as seções 8 e 9 trazem validação cruzada por paciente (variação da AUC entre divisões) e sensibilidade, especificidade e matriz de confusão no limiar escolhido no treino.

## 1. Configuração

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
RAIZ = Path.cwd()
MIMIC = RAIZ / "data" / "mimic-iv"
PASTA = RAIZ / "artefatos_24h_ampliado"
BASE = PASTA / "tabela_24h_ampliada.csv"
CACHE_IO = PASTA / "tabela_inout_24h.csv"
HORAS = 24
CORTES_DIAS = [3.55, 7.0]
N_BOOTSTRAP = 200

VASOPRESSORES = {221906: "noradrenalina", 221289: "adrenalina", 221749: "fenilefrina",
                 222315: "vasopressina", 221662: "dopamina"}
INOTROPICOS = {221653: "dobutamina", 221986: "milrinona"}
FUROSEMIDA_BOLUS, FUROSEMIDA_CONTINUA = 221794, 228340
ITENS_REMEDIO = set(VASOPRESSORES) | set(INOTROPICOS) | {FUROSEMIDA_BOLUS, FUROSEMIDA_CONTINUA}
CATEGORIAS_FORA_DO_VOLUME = {"14-Oral/Gastric Intake", "16-Pre Admission/Non-ICU"}
CATEGORIA_ORAL = "14-Oral/Gastric Intake"

# outputevents: urina (sonda, micção, cateter, nefrostomia, centro cirúrgico) e itens de irrigação a excluir
ITENS_URINA = {226559, 226560, 226561, 226567, 226584, 226563, 226564, 226565, 226557, 226558, 226627, 226631}
ITENS_IRRIGACAO = {227488, 227489}
VALOR_MAX_REGISTRO_ML = 5000.0

BASE_NUM = [
    "idade", "pam_24h", "frequencia_cardiaca_24h", "frequencia_respiratoria_24h", "spo2_24h",
    "creatinina_24h", "ureia_24h", "sodio_24h", "potassio_24h", "glicose_24h", "ptt_24h", "feve",
]
BASE_BIN = ["feve_ausente"]
NOVAS_NUM = [
    "temperatura_24h", "gcs_min_24h", "fio2_max_24h", "peep_max_24h",
    "lactato_max_24h", "bicarbonato_min_24h", "leucocitos_max_24h", "hemoglobina_min_24h",
    "plaquetas_min_24h", "inr_max_24h", "bilirrubina_max_24h", "nt_probnp_max_24h",
    "troponina_t_max_24h", "ph_min_24h", "pco2_max_24h",
]
NOVAS_BIN = ["ventilacao_invasiva_24h", "lactato_medido", "nt_probnp_medido", "troponina_medida", "gasometria_medida"]
IO_NUM = [
    "peso_kg", "n_vasopressores_24h", "noradrenalina_taxa_max_24h", "furosemida_dose_total_24h",
    "volume_infundido_24h", "ingestao_oral_ml", "diurese_total_24h", "diurese_ml_kg_h_24h",
    "saida_total_24h", "balanco_hidrico_24h", "balanco_ml_kg_24h",
]
IO_BIN = [
    "vasopressor_24h", "noradrenalina_24h", "vasopressina_24h", "adrenalina_24h", "fenilefrina_24h",
    "dopamina_24h", "inotropico_24h", "diuretico_alca_24h", "furosemida_infusao_continua_24h",
    "oliguria_24h", "diurese_medida",
]
CATEGORICAS = ["genero", "tipo_admissao"]

assert BASE.exists(), "Rode antes o caderno variaveis_24h_e_coorte_yang.ipynb."
for nome in ["inputevents", "outputevents"]:
    assert (MIMIC / f"{nome}.csv.gz").exists() or (MIMIC / f"{nome}.csv").exists(), f"falta {nome}"
print("Pronto.")

## 2. Leitura de `inputevents` e `outputevents` na janela de 24 horas

- **Infusões** que começam antes ou terminam depois da janela contam só a parte que cai dentro, proporcionalmente ao tempo.
- **Volume infundido:** só itens em mL. Ficam de fora a ingestão oral (contada à parte) e os volumes "pré-admissão" (dados antes da UTI).
- **Diurese:** itens de urina em mL. Registros negativos ou acima de 5.000 mL em uma só leitura são tratados como erro.
- **Balanço hídrico** = infundido + ingestão oral − todas as saídas registradas. Se não há nenhuma saída registrada, o balanço fica ausente (não zero).

In [ ]:
def achar(nome):
    for caminho in (MIMIC / f"{nome}.csv.gz", MIMIC / f"{nome}.csv"):
        if caminho.exists():
            return caminho
    raise FileNotFoundError(nome)


def varrer_inputevents(janela):
    inicio = janela.set_index("stay_id")["intime"]
    ids = set(inicio.index.astype(np.int64))
    colunas = ["stay_id", "starttime", "endtime", "itemid", "amount", "amountuom", "rate", "rateuom",
               "ordercategoryname", "patientweight"]
    partes, lidas = [], 0
    for bloco in pd.read_csv(achar("inputevents"), usecols=colunas, chunksize=1_000_000, low_memory=False):
        lidas += len(bloco)
        bloco = bloco[bloco["stay_id"].isin(ids)].copy()
        if bloco.empty:
            continue
        bloco["starttime"] = pd.to_datetime(bloco["starttime"], errors="coerce")
        bloco["endtime"] = pd.to_datetime(bloco["endtime"], errors="coerce")
        bloco = bloco.dropna(subset=["starttime", "endtime"])
        entrada = bloco["stay_id"].map(inicio)
        fim = entrada + pd.Timedelta(hours=HORAS)
        duracao = (bloco["endtime"] - bloco["starttime"]).dt.total_seconds()
        fim_na_janela = bloco["endtime"].where(bloco["endtime"] < fim, fim)
        inicio_na_janela = bloco["starttime"].where(bloco["starttime"] > entrada, entrada)
        sobreposto = (fim_na_janela - inicio_na_janela).dt.total_seconds().clip(lower=0)
        pontual = (bloco["starttime"] >= entrada) & (bloco["starttime"] < fim)
        fracao = np.where(duracao > 0, sobreposto / duracao.where(duracao > 0, 1.0), pontual.astype(float))
        bloco["fracao"] = fracao
        bloco = bloco[bloco["fracao"] > 0]
        bloco["amount_janela"] = bloco["amount"] * bloco["fracao"]
        partes.append(bloco[["stay_id", "itemid", "amount_janela", "amountuom", "rate", "rateuom",
                             "ordercategoryname", "patientweight"]])
        print(f"  inputevents: {lidas:,} linhas lidas", flush=True)
    return pd.concat(partes, ignore_index=True)


def resumir_inputevents(bruto, ids_estadia):
    saida = pd.DataFrame(index=pd.Index(ids_estadia, name="stay_id"))
    remedios = bruto[bruto["itemid"].isin(ITENS_REMEDIO)]

    for item, nome in VASOPRESSORES.items():
        saida[f"{nome}_24h"] = saida.index.isin(set(remedios.loc[remedios["itemid"] == item, "stay_id"])).astype(int)
    saida["vasopressor_24h"] = saida[[f"{n}_24h" for n in VASOPRESSORES.values()]].max(axis=1)
    saida["n_vasopressores_24h"] = saida[[f"{n}_24h" for n in VASOPRESSORES.values()]].sum(axis=1)
    inotropicos = set(remedios.loc[remedios["itemid"].isin(INOTROPICOS), "stay_id"])
    saida["inotropico_24h"] = saida.index.isin(inotropicos).astype(int)

    nora = remedios[(remedios["itemid"] == 221906) & (remedios["rateuom"] == "mcg/kg/min")
                    & (remedios["rate"] > 0) & (remedios["rate"] <= 5)]
    saida["noradrenalina_taxa_max_24h"] = nora.groupby("stay_id")["rate"].max()
    saida["noradrenalina_taxa_max_24h"] = saida["noradrenalina_taxa_max_24h"].fillna(0.0)

    furo = remedios[remedios["itemid"].isin([FUROSEMIDA_BOLUS, FUROSEMIDA_CONTINUA]) & (remedios["amountuom"] == "mg")]
    saida["diuretico_alca_24h"] = saida.index.isin(set(furo["stay_id"])).astype(int)
    dose = furo[(furo["amount_janela"] >= 0)].groupby("stay_id")["amount_janela"].sum().clip(upper=2000)
    saida["furosemida_dose_total_24h"] = dose
    saida["furosemida_dose_total_24h"] = saida["furosemida_dose_total_24h"].fillna(0.0)
    continua = set(furo.loc[furo["itemid"] == FUROSEMIDA_CONTINUA, "stay_id"])
    saida["furosemida_infusao_continua_24h"] = saida.index.isin(continua).astype(int)

    ml = bruto[(bruto["amountuom"] == "mL") & (bruto["amount_janela"] >= 0)]
    iv = ml[~ml["ordercategoryname"].isin(CATEGORIAS_FORA_DO_VOLUME)]
    saida["volume_infundido_24h"] = iv.groupby("stay_id")["amount_janela"].sum().clip(upper=30000)
    saida["volume_infundido_24h"] = saida["volume_infundido_24h"].fillna(0.0)
    oral = ml[ml["ordercategoryname"] == CATEGORIA_ORAL]
    saida["ingestao_oral_ml"] = oral.groupby("stay_id")["amount_janela"].sum().clip(upper=10000)
    saida["ingestao_oral_ml"] = saida["ingestao_oral_ml"].fillna(0.0)

    peso = bruto[(bruto["patientweight"] >= 25) & (bruto["patientweight"] <= 350)]
    saida["peso_kg"] = peso.groupby("stay_id")["patientweight"].median()
    return saida


def varrer_outputevents(janela):
    inicio = janela.set_index("stay_id")["intime"]
    ids = set(inicio.index.astype(np.int64))
    partes = []
    for bloco in pd.read_csv(achar("outputevents"), usecols=["stay_id", "charttime", "itemid", "value", "valueuom"],
                             chunksize=1_000_000, low_memory=False):
        bloco = bloco[bloco["stay_id"].isin(ids) & (bloco["valueuom"] == "mL")
                      & ~bloco["itemid"].isin(ITENS_IRRIGACAO)].copy()
        bloco["value"] = pd.to_numeric(bloco["value"], errors="coerce")
        bloco = bloco[(bloco["value"] >= 0) & (bloco["value"] <= VALOR_MAX_REGISTRO_ML)]
        bloco["charttime"] = pd.to_datetime(bloco["charttime"], errors="coerce")
        entrada = bloco["stay_id"].map(inicio)
        dentro = (bloco["charttime"] >= entrada) & (bloco["charttime"] < entrada + pd.Timedelta(hours=HORAS))
        partes.append(bloco.loc[dentro, ["stay_id", "itemid", "value"]])
    print("  outputevents lido", flush=True)
    return pd.concat(partes, ignore_index=True)


def resumir_outputevents(bruto, ids_estadia):
    saida = pd.DataFrame(index=pd.Index(ids_estadia, name="stay_id"))
    urina = bruto[bruto["itemid"].isin(ITENS_URINA)]
    saida["diurese_total_24h"] = urina.groupby("stay_id")["value"].sum()
    saida["saida_total_24h"] = bruto.groupby("stay_id")["value"].sum()
    saida["diurese_medida"] = saida["diurese_total_24h"].notna().astype(int)
    return saida


def montar_io(base):
    janela = base[["stay_id", "intime"]].drop_duplicates("stay_id")
    ids_estadia = janela["stay_id"].astype(np.int64).to_numpy()
    print("Lendo inputevents...", flush=True)
    entrada = resumir_inputevents(varrer_inputevents(janela), ids_estadia)
    print("Lendo outputevents...", flush=True)
    saida = resumir_outputevents(varrer_outputevents(janela), ids_estadia)
    io = entrada.join(saida)
    io["diurese_ml_kg_h_24h"] = io["diurese_total_24h"] / (io["peso_kg"] * HORAS)
    io["oliguria_24h"] = (io["diurese_ml_kg_h_24h"] < 0.5).astype(int)
    io.loc[io["diurese_ml_kg_h_24h"].isna(), "oliguria_24h"] = 0
    io["balanco_hidrico_24h"] = io["volume_infundido_24h"] + io["ingestao_oral_ml"] - io["saida_total_24h"]
    io["balanco_ml_kg_24h"] = io["balanco_hidrico_24h"] / io["peso_kg"]
    return io.reset_index()

In [ ]:
base = pd.read_csv(BASE, parse_dates=["intime"])
if CACHE_IO.exists():
    io = pd.read_csv(CACHE_IO)
    print("Variáveis de inputevents e outputevents lidas do cache.")
else:
    io = montar_io(base)
    io.to_csv(CACHE_IO, index=False, encoding="utf-8-sig")
    print("Calculadas e salvas no cache.")

tabela = base.merge(io, on="stay_id", how="left")
tabela["lactato_medido"] = tabela["lactato_max_24h"].notna().astype(int)
tabela["nt_probnp_medido"] = tabela["nt_probnp_max_24h"].notna().astype(int)
tabela["troponina_medida"] = tabela["troponina_t_max_24h"].notna().astype(int)
tabela["gasometria_medida"] = tabela["ph_min_24h"].notna().astype(int)
for coluna in ["ventilacao_invasiva_24h"] + [c for c in IO_BIN if c != "diurese_medida"] + ["diurese_medida"]:
    tabela[coluna] = tabela[coluna].fillna(0).astype(int)
tabela = tabela.dropna(subset=["los_dias", "subject_id"])
tabela = tabela[tabela["los_dias"] > 0].copy()
print(f"Estadias: {len(tabela):,} | pacientes: {tabela['subject_id'].nunique():,}")
print(f"Saem da UTI antes de 24 h: {(tabela['los_dias'] < 1).sum():,} ({100 * (tabela['los_dias'] < 1).mean():.1f}%)")

## 3. Conferência das variáveis novas

Valores típicos esperados: noradrenalina entre 0,05 e 0,3 mcg/kg/min; furosemida em dezenas de mg; diurese perto de 1 mL/kg/h; peso entre 60 e 90 kg; balanço positivo nas primeiras 24 horas.

In [ ]:
conferencia = []
for coluna in IO_NUM:
    serie = tabela[coluna]
    conferencia.append({
        "variavel": coluna, "pct_presente": round(100 * serie.notna().mean(), 1),
        "p5": round(serie.quantile(0.05), 2), "mediana": round(serie.median(), 2), "p95": round(serie.quantile(0.95), 2),
    })
display(pd.DataFrame(conferencia))
uso = pd.DataFrame({"pct_das_estadias": [round(100 * tabela[c].mean(), 1) for c in IO_BIN]}, index=IO_BIN)
display(uso)

d_items = MIMIC / "d_items.csv.gz"
if d_items.exists():
    nomes = pd.read_csv(d_items, usecols=["itemid", "label"]).set_index("itemid")["label"]
    codigos = list(VASOPRESSORES) + list(INOTROPICOS) + [FUROSEMIDA_BOLUS, FUROSEMIDA_CONTINUA] + sorted(ITENS_URINA)
    display(nomes.reindex(codigos).to_frame("nome_no_d_items"))
else:
    print("d_items.csv.gz não está em data/mimic-iv/. Conferência dos nomes não feita.")

## 4. Coortes

- **Todas, como antes:** todas as estadias de UTI com IC (referência).
- **Ainda na UTI às 24 h:** `los_dias ≥ 1`.
- **Yang:** primeira UTI de cada paciente (pela data de entrada, dentro da coorte de IC) e sem óbito hospitalar. A versão com filtro de 24 horas aplica `los_dias ≥ 1` depois.

In [ ]:
todas = tabela.copy()
na_uti = tabela[tabela["los_dias"] >= 1.0].copy()
primeira = tabela.sort_values("intime").drop_duplicates("subject_id", keep="first")
yang_todas = primeira[primeira["hospital_expire_flag"] == 0]
yang_na_uti = yang_todas[yang_todas["los_dias"] >= 1.0].copy()

resumo = []
for nome, b in [("Todas (como antes)", todas), ("Ainda na UTI às 24 h", na_uti), ("Yang, ainda na UTI às 24 h", yang_na_uti)]:
    linha = {"coorte": nome, "estadias": len(b), "pacientes": b["subject_id"].nunique(),
             "los_mediano_dias": round(b["los_dias"].median(), 2)}
    for corte in CORTES_DIAS:
        linha[f"pct_mais_de_{corte}"] = round(100 * (b["los_dias"] > corte).mean(), 1)
    resumo.append(linha)
display(pd.DataFrame(resumo))

## 5. Preparação e modelos

In [ ]:
def preparar(base, colunas_num, colunas_bin):
    divisor = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    i_treino, i_teste = next(divisor.split(base, groups=base["subject_id"]))
    treino, teste = base.iloc[i_treino].copy(), base.iloc[i_teste].copy()
    assert not set(treino["subject_id"]) & set(teste["subject_id"]), "paciente repetido"

    for c in CATEGORICAS:
        treino[c] = treino[c].astype("string").fillna("desconhecido")
        teste[c] = teste[c].astype("string").fillna("desconhecido")
    try:
        codificador = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    except TypeError:
        codificador = OneHotEncoder(sparse=False, handle_unknown="ignore", dtype=np.float64)
    d_treino = codificador.fit_transform(treino[CATEGORICAS])
    nomes = codificador.get_feature_names_out(CATEGORICAS)
    d_treino = pd.DataFrame(d_treino, columns=nomes, index=treino.index)
    d_teste = pd.DataFrame(codificador.transform(teste[CATEGORICAS]), columns=nomes, index=teste.index)

    imputador = SimpleImputer(strategy="median")
    n_treino = pd.DataFrame(imputador.fit_transform(treino[colunas_num]), columns=colunas_num, index=treino.index)
    n_teste = pd.DataFrame(imputador.transform(teste[colunas_num]), columns=colunas_num, index=teste.index)
    x_treino = pd.concat([n_treino, treino[colunas_bin].astype(float), d_treino], axis=1)
    x_teste = pd.concat([n_teste, teste[colunas_bin].astype(float), d_teste], axis=1)
    x_teste = x_teste.reindex(columns=x_treino.columns, fill_value=0.0)
    proibidas = [c for c in x_treino.columns if "los" in c or "icd" in c.lower() or "diag" in c.lower()]
    assert not proibidas, f"coluna proibida em X: {proibidas}"
    return treino, teste, x_treino, x_teste


def intervalo_auc(y, prob, grupos, repeticoes=N_BOOTSTRAP):
    grupos = np.asarray(grupos)
    ordem = np.argsort(grupos, kind="stable")
    _, inicio = np.unique(grupos[ordem], return_index=True)
    por_paciente = np.split(ordem, inicio[1:])
    sorteio = np.random.default_rng(RANDOM_STATE)
    aucs = []
    for _ in range(repeticoes):
        escolha = sorteio.integers(0, len(por_paciente), len(por_paciente))
        indices = np.concatenate([por_paciente[i] for i in escolha])
        if y[indices].min() == y[indices].max():
            continue
        aucs.append(roc_auc_score(y[indices], prob[indices]))
    return tuple(np.percentile(aucs, [2.5, 97.5]))


def avaliar_variante(rotulo, base, colunas_num, colunas_bin):
    treino, teste, x_treino, x_teste = preparar(base, colunas_num, colunas_bin)
    linhas, guardado = [], {}
    for corte in CORTES_DIAS:
        y_treino = (treino["los_dias"] > corte).astype(int).to_numpy()
        y_teste = (teste["los_dias"] > corte).astype(int).to_numpy()
        modelo = XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, scale_pos_weight=float((y_treino == 0).sum() / (y_treino == 1).sum()),
            objective="binary:logistic", eval_metric="auc", n_jobs=1, random_state=RANDOM_STATE,
        )
        modelo.fit(x_treino, y_treino)
        prob = modelo.predict_proba(x_teste)[:, 1]
        baixo, alto = intervalo_auc(y_teste, prob, teste["subject_id"])
        linhas.append({
            "variante": rotulo, "tarefa": f"mais de {corte} dias", "AUC": roc_auc_score(y_teste, prob),
            "AUC_IC95_baixo": baixo, "AUC_IC95_alto": alto, "AUC_PR": average_precision_score(y_teste, prob),
            "prevalencia": y_teste.mean(), "n_teste": len(teste),
        })
        if corte == CORTES_DIAS[0]:
            guardado = {"modelo": modelo, "x_teste": x_teste}
    regressor = XGBRegressor(
        n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, objective="reg:squarederror", n_jobs=1, random_state=RANDOM_STATE,
    )
    regressor.fit(x_treino, np.log1p(treino["los_dias"].to_numpy()))
    previsto_log = regressor.predict(x_teste)
    previsto = np.clip(np.expm1(previsto_log), 0, None)
    linhas.append({
        "variante": rotulo, "tarefa": "regressão (dias)",
        "MAE": mean_absolute_error(teste["los_dias"], previsto),
        "R2": r2_score(teste["los_dias"], previsto),
        "R2_log": r2_score(np.log1p(teste["los_dias"]), previsto_log),
        "n_teste": len(teste),
    })
    return linhas, guardado

In [ ]:
conjuntos = {
    "antigas": (BASE_NUM, BASE_BIN),
    "mais chart/labs": (BASE_NUM + NOVAS_NUM, BASE_BIN + NOVAS_BIN),
    "mais inputevents/outputevents": (BASE_NUM + NOVAS_NUM + IO_NUM, BASE_BIN + NOVAS_BIN + IO_BIN),
}
plano = [
    ("Ref: todas, chart/labs", todas, conjuntos["mais chart/labs"]),
    ("E: na UTI às 24 h, antigas", na_uti, conjuntos["antigas"]),
    ("F: na UTI às 24 h, + chart/labs", na_uti, conjuntos["mais chart/labs"]),
    ("G: na UTI às 24 h, + input/output", na_uti, conjuntos["mais inputevents/outputevents"]),
    ("E-Y: Yang, antigas", yang_na_uti, conjuntos["antigas"]),
    ("F-Y: Yang, + chart/labs", yang_na_uti, conjuntos["mais chart/labs"]),
    ("G-Y: Yang, + input/output", yang_na_uti, conjuntos["mais inputevents/outputevents"]),
]
linhas_total, guardados = [], {}
for rotulo, coorte, (num, bin_) in plano:
    print(f"Treinando {rotulo}...", flush=True)
    linhas, guardado = avaliar_variante(rotulo, coorte, num, bin_)
    linhas_total.extend(linhas)
    guardados[rotulo] = guardado
resultados = pd.DataFrame(linhas_total)
resultados.to_csv(PASTA / "resultados_input_output.csv", index=False, encoding="utf-8-sig")

## 6. Resultados

In [ ]:
classificacao = resultados[resultados["tarefa"].str.startswith("mais de")].copy()
classificacao["AUC (IC 95%)"] = classificacao.apply(
    lambda l: f"{l['AUC']:.3f} ({l['AUC_IC95_baixo']:.3f} a {l['AUC_IC95_alto']:.3f})", axis=1)
classificacao["prevalência"] = (100 * classificacao["prevalencia"]).round(1).astype(str) + "%"
display(classificacao[["variante", "tarefa", "n_teste", "prevalência", "AUC (IC 95%)", "AUC_PR"]].round(3).reset_index(drop=True))
display(resultados[resultados["tarefa"] == "regressão (dias)"][["variante", "n_teste", "MAE", "R2", "R2_log"]].round(3).reset_index(drop=True))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for eixo, corte in zip(eixos, CORTES_DIAS):
    parte = classificacao[classificacao["tarefa"] == f"mais de {corte} dias"].reset_index(drop=True)
    y = np.arange(len(parte))[::-1]
    erro = [parte["AUC"] - parte["AUC_IC95_baixo"], parte["AUC_IC95_alto"] - parte["AUC"]]
    eixo.errorbar(parte["AUC"], y, xerr=erro, fmt="o", color="#1f4e79", capsize=4)
    eixo.axvline(0.5, color="gray", linestyle="--", linewidth=1)
    eixo.set_yticks(y)
    eixo.set_yticklabels(parte["variante"])
    eixo.set_title(f"Estadia > {corte} dias")
    eixo.set_xlabel("AUC no teste, com IC 95%")
    eixo.set_xlim(0.45, 1.0)
plt.tight_layout()
plt.savefig(PASTA / "auc_input_output.png", dpi=150)
plt.show()

## 7. Que variáveis pesam mais (SHAP, variante G, estadia > 3,55 dias)

As variáveis de `inputevents` e `outputevents` aparecem com ★.

In [ ]:
rotulo = "G: na UTI às 24 h, + input/output"
modelo, x_teste = guardados[rotulo]["modelo"], guardados[rotulo]["x_teste"]
amostra = x_teste.sample(n=min(2000, len(x_teste)), random_state=RANDOM_STATE)
valores = np.asarray(shap.TreeExplainer(modelo)(amostra).values)
pesos = (
    pd.DataFrame({"variavel": amostra.columns, "shap_medio_abs": np.abs(valores).mean(axis=0)})
    .sort_values("shap_medio_abs", ascending=False).reset_index(drop=True)
)
pesos["novo_io"] = pesos["variavel"].isin(IO_NUM + IO_BIN)
pesos.to_csv(PASTA / "shap_variante_G.csv", index=False, encoding="utf-8-sig")
topo = pesos.head(15).iloc[::-1]
rotulos = [("★ " if n else "") + v for v, n in zip(topo["variavel"], topo["novo_io"])]
plt.figure(figsize=(8, 6))
plt.barh(rotulos, topo["shap_medio_abs"], color=["#c0504d" if n else "#1f4e79" for n in topo["novo_io"]])
plt.xlabel("SHAP médio absoluto")
plt.title("Variante G, estadia > 3,55 dias")
plt.tight_layout()
plt.savefig(PASTA / "shap_variante_G.png", dpi=150)
plt.show()
display(pesos.head(12))

## 8. Validação cruzada por paciente (5 partes)

A seção 6 usa uma única divisão 80/20. Aqui cada paciente cai em uma de 5 partes, e cada parte vira teste uma vez, com o treino nas outras 4. Isso mostra **quanto a AUC varia de uma divisão para outra** e se o ganho de cada grupo de variáveis aparece em todas as partes.

- As partes são formadas por paciente (o mesmo paciente nunca fica no treino e no teste da mesma rodada).
- As partes são as mesmas para as variantes da mesma coorte, o que permite comparar os ganhos pareados.
- Imputação e one-hot são ajustados só no treino de cada rodada.
- Aplicado às variantes E, F e G, na coorte "ainda na UTI às 24 h" e na de Yang.

In [ ]:
N_PARTICOES = 5


def preparar_particao(treino, teste, colunas_num, colunas_bin):
    treino, teste = treino.copy(), teste.copy()
    for c in CATEGORICAS:
        treino[c] = treino[c].astype("string").fillna("desconhecido")
        teste[c] = teste[c].astype("string").fillna("desconhecido")
    try:
        codificador = OneHotEncoder(sparse_output=False, handle_unknown="ignore", dtype=np.float64)
    except TypeError:
        codificador = OneHotEncoder(sparse=False, handle_unknown="ignore", dtype=np.float64)
    d_treino = codificador.fit_transform(treino[CATEGORICAS])
    nomes = codificador.get_feature_names_out(CATEGORICAS)
    d_treino = pd.DataFrame(d_treino, columns=nomes, index=treino.index)
    d_teste = pd.DataFrame(codificador.transform(teste[CATEGORICAS]), columns=nomes, index=teste.index)
    imputador = SimpleImputer(strategy="median")
    n_treino = pd.DataFrame(imputador.fit_transform(treino[colunas_num]), columns=colunas_num, index=treino.index)
    n_teste = pd.DataFrame(imputador.transform(teste[colunas_num]), columns=colunas_num, index=teste.index)
    x_treino = pd.concat([n_treino, treino[colunas_bin].astype(float), d_treino], axis=1)
    x_teste = pd.concat([n_teste, teste[colunas_bin].astype(float), d_teste], axis=1)
    return x_treino, x_teste.reindex(columns=x_treino.columns, fill_value=0.0)


def atribuir_particoes(base, n=N_PARTICOES, semente=RANDOM_STATE):
    pacientes = np.random.default_rng(semente).permutation(base["subject_id"].unique())
    mapa = {paciente: i % n for i, paciente in enumerate(pacientes)}
    return base["subject_id"].map(mapa).to_numpy()


def novo_classificador(y_treino):
    return XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=5, scale_pos_weight=float((y_treino == 0).sum() / (y_treino == 1).sum()),
        objective="binary:logistic", eval_metric="auc", n_jobs=1, random_state=RANDOM_STATE,
    )


def validar(rotulo, base, colunas_num, colunas_bin):
    particao = atribuir_particoes(base)
    linhas = []
    for k in range(N_PARTICOES):
        treino, teste = base[particao != k], base[particao == k]
        assert not set(treino["subject_id"]) & set(teste["subject_id"]), "paciente repetido"
        x_treino, x_teste = preparar_particao(treino, teste, colunas_num, colunas_bin)
        for corte in CORTES_DIAS:
            y_treino = (treino["los_dias"] > corte).astype(int).to_numpy()
            y_teste = (teste["los_dias"] > corte).astype(int).to_numpy()
            modelo = novo_classificador(y_treino).fit(x_treino, y_treino)
            prob = modelo.predict_proba(x_teste)[:, 1]
            linhas.append({"variante": rotulo, "tarefa": f"mais de {corte} dias", "particao": k,
                           "AUC": roc_auc_score(y_teste, prob), "AUC_PR": average_precision_score(y_teste, prob)})
        regressor = XGBRegressor(
            n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, objective="reg:squarederror", n_jobs=1, random_state=RANDOM_STATE,
        ).fit(x_treino, np.log1p(treino["los_dias"].to_numpy()))
        linhas.append({"variante": rotulo, "tarefa": "regressão (log)", "particao": k,
                       "R2_log": r2_score(np.log1p(teste["los_dias"]), regressor.predict(x_teste))})
    return linhas


plano_cv = [
    ("E", "todas", na_uti, conjuntos["antigas"]),
    ("F", "todas", na_uti, conjuntos["mais chart/labs"]),
    ("G", "todas", na_uti, conjuntos["mais inputevents/outputevents"]),
    ("E", "Yang", yang_na_uti, conjuntos["antigas"]),
    ("F", "Yang", yang_na_uti, conjuntos["mais chart/labs"]),
    ("G", "Yang", yang_na_uti, conjuntos["mais inputevents/outputevents"]),
]
linhas_cv = []
for variante, coorte_nome, coorte, (num, bin_) in plano_cv:
    print(f"Validação cruzada: {variante} ({coorte_nome})...", flush=True)
    for linha in validar(variante, coorte, num, bin_):
        linhas_cv.append({"coorte": coorte_nome, **linha})
cv = pd.DataFrame(linhas_cv)
cv.to_csv(PASTA / "validacao_cruzada_por_particao.csv", index=False, encoding="utf-8-sig")

In [ ]:
resumo_cv = (
    cv.melt(id_vars=["coorte", "variante", "tarefa", "particao"], value_vars=["AUC", "AUC_PR", "R2_log"])
    .dropna(subset=["value"])
    .groupby(["coorte", "tarefa", "variante", "variable"])["value"]
    .agg(media="mean", desvio="std", minimo="min", maximo="max")
    .reset_index()
)
resumo_cv.to_csv(PASTA / "validacao_cruzada_resumo.csv", index=False, encoding="utf-8-sig")
principal = resumo_cv[resumo_cv["variable"].isin(["AUC", "R2_log"])].round(3)
display(principal.rename(columns={"variable": "metrica"}).reset_index(drop=True))

# Ganho pareado entre variantes, partição a partição
ganhos = []
for coorte_nome in ["todas", "Yang"]:
    for tarefa in ["mais de 3.55 dias", "mais de 7.0 dias"]:
        parte = cv[(cv["coorte"] == coorte_nome) & (cv["tarefa"] == tarefa)].pivot(
            index="particao", columns="variante", values="AUC")
        for de, para in [("E", "F"), ("F", "G"), ("E", "G")]:
            delta = parte[para] - parte[de]
            ganhos.append({
                "coorte": coorte_nome, "tarefa": tarefa, "comparacao": f"{de} → {para}",
                "ganho_medio_AUC": delta.mean(), "desvio": delta.std(),
                "particoes_com_ganho": f"{int((delta > 0).sum())} de {len(delta)}",
            })
ganhos = pd.DataFrame(ganhos)
ganhos.to_csv(PASTA / "validacao_cruzada_ganhos.csv", index=False, encoding="utf-8-sig")
display(ganhos.round(3))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for eixo, tarefa in zip(eixos, ["mais de 3.55 dias", "mais de 7.0 dias"]):
    for deslocamento, (coorte_nome, cor) in zip([-0.12, 0.12], [("todas", "#1f4e79"), ("Yang", "#c0504d")]):
        parte = resumo_cv[(resumo_cv["coorte"] == coorte_nome) & (resumo_cv["tarefa"] == tarefa)
                          & (resumo_cv["variable"] == "AUC")].set_index("variante").loc[["E", "F", "G"]]
        y = np.arange(3)[::-1] + deslocamento
        eixo.errorbar(parte["media"], y, xerr=parte["desvio"], fmt="o", color=cor, capsize=4, label=coorte_nome)
    eixo.set_yticks(np.arange(3)[::-1])
    eixo.set_yticklabels(["E: antigas", "F: + chart/labs", "G: + input/output"])
    eixo.set_title(tarefa.replace("mais de", "Estadia >").replace(" dias", " dias"))
    eixo.set_xlabel("AUC média nas 5 partições (± desvio)")
    eixo.legend(loc="lower right")
plt.tight_layout()
plt.savefig(PASTA / "validacao_cruzada_auc.png", dpi=150)
plt.show()

## 9. Sensibilidade, especificidade e matriz de confusão

AUC não diz como o modelo se comporta quando é preciso decidir "longa" ou "curta". Aqui a decisão usa um **limiar escolhido só no treino** (previsões fora da amostra em 5 partições por paciente), e o teste só mede o resultado.

- **Limiar de Youden:** maximiza sensibilidade + especificidade − 1.
- **Limiar de sensibilidade 80%:** o mais alto que ainda capta 80% das estadias longas. Mostra o custo em falsos alarmes.
- **Valor preditivo positivo (VPP):** de quem o modelo marca como longo, quantos são longos. **Valor preditivo negativo (VPN):** de quem marca como curto, quantos são curtos.
- Variante G, nas duas coortes, no mesmo teste de 20% da seção 5.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve


def limiar_youden(y, prob):
    falsos, verdadeiros, limiares = roc_curve(y, prob)
    return float(limiares[np.argmax(verdadeiros - falsos)])


def limiar_sensibilidade(y, prob, alvo=0.80):
    falsos, verdadeiros, limiares = roc_curve(y, prob)
    candidatos = np.where(verdadeiros >= alvo)[0]
    return float(limiares[candidatos[0]])


def decisao(y, prob, limiar):
    predito = (prob >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predito).ravel()
    sens, espec = tp / (tp + fn), tn / (tn + fp)
    return {
        "sensibilidade": sens, "especificidade": espec, "VPP": tp / (tp + fp), "VPN": tn / (tn + fn),
        "acuracia_balanceada": (sens + espec) / 2, "limiar": limiar,
        "VP": tp, "FP": fp, "FN": fn, "VN": tn,
    }


def decisoes(rotulo, coorte, colunas_num, colunas_bin):
    treino, teste, x_treino, x_teste = preparar(coorte, colunas_num, colunas_bin)
    particao = atribuir_particoes(treino)
    linhas = []
    for corte in CORTES_DIAS:
        y_treino = (treino["los_dias"] > corte).astype(int).to_numpy()
        y_teste = (teste["los_dias"] > corte).astype(int).to_numpy()
        fora = np.zeros(len(treino))
        for k in range(N_PARTICOES):
            dentro = particao != k
            fora[~dentro] = novo_classificador(y_treino[dentro]).fit(
                x_treino[dentro], y_treino[dentro]).predict_proba(x_treino[~dentro])[:, 1]
        modelo = novo_classificador(y_treino).fit(x_treino, y_treino)
        prob = modelo.predict_proba(x_teste)[:, 1]
        for nome, limiar in [("Youden", limiar_youden(y_treino, fora)),
                             ("sensibilidade 80%", limiar_sensibilidade(y_treino, fora))]:
            linhas.append({"variante": rotulo, "tarefa": f"mais de {corte} dias", "limiar_tipo": nome,
                           "prevalencia": y_teste.mean(), **decisao(y_teste, prob, limiar)})
    return linhas


linhas_dec = []
for rotulo, coorte in [("G (todas na UTI às 24 h)", na_uti), ("G-Y (Yang)", yang_na_uti)]:
    print(f"Decisões: {rotulo}...", flush=True)
    num, bin_ = conjuntos["mais inputevents/outputevents"]
    linhas_dec.extend(decisoes(rotulo, coorte, num, bin_))
decisoes_tabela = pd.DataFrame(linhas_dec)
decisoes_tabela.to_csv(PASTA / "decisao_sensibilidade_especificidade.csv", index=False, encoding="utf-8-sig")
colunas_mostrar = ["variante", "tarefa", "limiar_tipo", "prevalencia", "sensibilidade", "especificidade",
                   "VPP", "VPN", "acuracia_balanceada", "limiar"]
display(decisoes_tabela[colunas_mostrar].round(3))

In [ ]:
confusao = decisoes_tabela[decisoes_tabela["limiar_tipo"] == "Youden"][
    ["variante", "tarefa", "VN", "FP", "FN", "VP"]].rename(
    columns={"VN": "curta, marcada curta", "FP": "curta, marcada longa",
             "FN": "longa, marcada curta", "VP": "longa, marcada longa"})
display(confusao.reset_index(drop=True))

**Como ler as seções 8 e 9**

- Na validação cruzada, se o **desvio** entre partições for menor que o ganho médio e o ganho aparecer em **5 de 5 partições**, o ganho é confiável. Se aparecer em 3 de 5 ou menos, é incerto.
- No limiar de Youden, sensibilidade e especificidade ficam parecidas. No limiar de **sensibilidade 80%**, a especificidade cai: é o custo de captar mais estadias longas.
- **VPP baixo** quer dizer muitos falsos alarmes. Para 7 dias, a prevalência é baixa (cerca de 14 a 17%), então o VPP tende a ser baixo mesmo com AUC razoável.

## 10. Como ler

- **Ref contra E:** mostra quanto da AUC antiga vinha de incluir pacientes que saem antes de 24 horas.
- **E, F e G:** mostram o ganho de cada grupo de variáveis na coorte correta (quem ainda está na UTI às 24 horas).
- **Versões -Y:** a coorte de Yang (primeira UTI e sem óbitos) com o mesmo filtro.
- Com o filtro de 24 horas, a estadia mínima passa a ser de 1 dia. Isso aumenta a fração de estadias longas e muda o ponto de comparação.
- A AUC antiga (0,76 a 0,79) não deve ser citada sem dizer que a coorte incluía quem saía da UTI antes de 24 horas.